In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Add project root to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

from src.utils.data_cleaning import clean_stock_code

# Read the data
cnn_df = pd.read_csv('../src/data/dataset/CNN_Model_Train_Data.csv')
ecommerce_df = pd.read_csv('../src/data/dataset/cleaned_ecommerce_data.csv')

# Clean stock code for cnn_df
cnn_df = clean_stock_code(cnn_df)

cnn_df.head()


In [ ]:
# Get unique descriptions for each stock code
product_descriptions = (ecommerce_df[['StockCode', 'Description']]
                      .drop_duplicates()
                      .dropna(subset=['Description']))


merged_df = pd.merge(
    cnn_df,
    product_descriptions,
    on='StockCode',
    how='left'
)

merged_df.head()

In [5]:
# Create directory structure for images
static_dir = Path('../static/images')
static_dir.mkdir(parents=True, exist_ok=True)

# Create product directories
for stock_code in merged_df['StockCode'].unique():
    product_dir = static_dir / stock_code
    product_dir.mkdir(exist_ok=True)

# Save prepared data
merged_df.to_csv('../src/data/dataset/cleaned/prepared_cnn_data.csv', index=False)

In [10]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the scraping function
from src.scripts.train_cnn_from_scratch import main

In [ ]:
# Run the training
try:
    history = main(
        project_root_str=project_root,
        batch_size=32,
        num_epochs=50,
        learning_rate=0.001
    )
    print("\nTraining completed successfully!")
    print(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
except Exception as e:
    print(f"Error occurred: {str(e)}")
    raise

In [ ]:
#Alternative method
# !python src/scripts/train_cnn_from_scratch.py . --batch_size 32 --num_epochs 50 --learning_rate 0.001